# CKD — ANOVA + Rank Fusion: Primary Analysis vs. Sensitivity Analysis

This notebook runs the **same pipeline twice** on the same underlying CKD data, so the two runs can be compared directly:

- **Primary Analysis**: the full 400-row dataset, with missing values handled by median (numerical) / mode (categorical) imputation, fit on training data only within each fold.
- **Sensitivity Analysis**: only the 158 complete-case rows (zero missing values anywhere), no imputation needed at all.

**Why this comparison matters, concretely**: dropping every row with a missing value would cost 60.5% of the dataset and visibly shifts the class balance (62.5%/37.5% CKD in the full data vs. 27.2%/72.8% in the complete cases) — a strong sign the missingness isn't random. If the Primary and Sensitivity results agree, that's real evidence the imputation approach isn't distorting the conclusions. If they disagree, that's equally real evidence — and tells you exactly where to be cautious.

**Methodology upgrade from the earlier `ckd_anova.ipynb`**: this version uses `doda.fusion.RankFusion` directly from the package — it's been implemented properly since then as genuine **Reciprocal Rank Fusion (RRF)**, `RRF(f) = 1/(k + math_rank(f)) + 1/(k + clinical_rank(f))`, the same algorithm from Cormack, Clarke & Buettcher (2009). No more locally-redefined fusion class needed.

**Also new**: feature-selection stability (25-run repeated stratified CV, Jaccard similarity) and CV predictive performance, both with **paired Wilcoxon signed-rank tests, Holm correction, and Cohen's d effect sizes** comparing ANOVA vs. DODA — following the same rigor established in the Heart Disease experiment, applied here to both Primary and Sensitivity data.

**Imbalance handling**: `class_weight="balanced"` / `scale_pos_weight` used throughout (unlike the Heart Disease notebook, which didn't need this) since CKD is meaningfully imbalanced (62.5/37.5) — consistent with how breast cancer's similar imbalance was handled earlier in this project.

In [ ]:
# =============================================================================
# STEP 1: LOAD AND CLEAN RAW DATA (shared by both analyses)
# =============================================================================

import pandas as pd
import numpy as np

df = pd.read_csv("../../data/raw/ckd.csv")
df = df.drop(columns=["id"])

# Known data-quality issues in this exact UCI file (see 01_ckd_eda.ipynb)
categorical_cols_raw = df.select_dtypes(include="object").columns
for col in categorical_cols_raw:
    df[col] = df[col].astype(str).str.strip()
    df[col] = df[col].replace({"nan": np.nan, "?": np.nan})

for col in ["pcv", "wc", "rc"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Encode categorical features and target
target_column = "target"

binary_maps = {
    "rbc":   {"normal": 0, "abnormal": 1},
    "pc":    {"normal": 0, "abnormal": 1},
    "pcc":   {"notpresent": 0, "present": 1},
    "ba":    {"notpresent": 0, "present": 1},
    "htn":   {"no": 0, "yes": 1},
    "dm":    {"no": 0, "yes": 1},
    "cad":   {"no": 0, "yes": 1},
    "appet": {"poor": 0, "good": 1},
    "pe":    {"no": 0, "yes": 1},
    "ane":   {"no": 0, "yes": 1},
}
for col, mapping in binary_maps.items():
    df[col] = df[col].map(mapping)

df[target_column] = df["classification"].map({"ckd": 1, "notckd": 0})
df = df.drop(columns=["classification"])

numerical_features = ["age", "bp", "sg", "al", "su", "bgr", "bu", "sc",
                       "sod", "pot", "hemo", "pcv", "wc", "rc"]
categorical_features = ["rbc", "pc", "pcc", "ba", "htn", "dm", "cad",
                         "appet", "pe", "ane"]

print("=" * 70)
print("CLEANED + ENCODED DATASET")
print("=" * 70)
print(f"Shape: {df.shape}")
display(df.head())

In [ ]:
# =============================================================================
# STEP 2: DEFINE THE TWO DATASETS — PRIMARY (imputed) vs SENSITIVITY (complete-case)
# =============================================================================

# --- Primary: full dataset, missing values handled by imputation later ---
X_primary = df.drop(columns=[target_column])
y_primary = df[target_column]

# --- Sensitivity: complete cases only, no imputation needed ---
df_complete = df.dropna()
X_sensitivity = df_complete.drop(columns=[target_column])
y_sensitivity = df_complete[target_column]

print("=" * 70)
print("PRIMARY vs SENSITIVITY DATASET SIZES")
print("=" * 70)
print(f"Primary (full, to be imputed) : {X_primary.shape[0]} rows")
print(f"Sensitivity (complete-case)    : {X_sensitivity.shape[0]} rows "
      f"({X_sensitivity.shape[0]/X_primary.shape[0]*100:.1f}% of primary)")

print("\nClass balance comparison:")
print("Primary:")
display((y_primary.value_counts(normalize=True) * 100).round(2))
print("Sensitivity:")
display((y_sensitivity.value_counts(normalize=True) * 100).round(2))

In [ ]:
#%pip uninstall -y doda

In [ ]:
#%pip install --no-cache-dir git+https://github.com/anandha-3679/DODA.git

In [ ]:
# =============================================================================
# STEP 3: SHARED HELPER FUNCTIONS
# Defined once, called twice (Primary and Sensitivity) — avoids duplicating
# ~150 lines of loop logic twice with only the input data differing.
# =============================================================================

from sklearn.model_selection import RepeatedStratifiedKFold, train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
)
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests

from doda import DODASelector
from doda.adapters.sklearn_adapter import SklearnAdapter
from doda.knowledge import JSONProvider
from doda.fusion import RankFusion

WEIGHTS_FILE = "../../../config/clinical_weights/ckd_clinical_weights.json"


def make_models(y_train):
    """Fresh model instances, class-imbalance-aware (CKD is ~63/37)."""
    return {
        "LR": LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42),
        "RF": RandomForestClassifier(n_estimators=300, class_weight="balanced",
                                      random_state=42, n_jobs=-1),
        "XGB": XGBClassifier(n_estimators=300, max_depth=3, learning_rate=0.05,
                              subsample=0.8, colsample_bytree=0.8, eval_metric="logloss",
                              scale_pos_weight=(y_train == 0).sum() / max((y_train == 1).sum(), 1),
                              random_state=42, n_jobs=-1)
    }


def impute_if_needed(X_train, X_test, do_impute):
    """Median/mode imputation fit on TRAIN only — skipped entirely for the
    complete-case (Sensitivity) dataset, since it has no missing values."""
    if not do_impute:
        return X_train.copy(), X_test.copy()

    num_cols = [c for c in numerical_features if c in X_train.columns]
    cat_cols = [c for c in categorical_features if c in X_train.columns]

    num_imp = SimpleImputer(strategy="median")
    cat_imp = SimpleImputer(strategy="most_frequent")

    X_train_i, X_test_i = X_train.copy(), X_test.copy()
    X_train_i[num_cols] = num_imp.fit_transform(X_train[num_cols])
    X_test_i[num_cols] = num_imp.transform(X_test[num_cols])
    X_train_i[cat_cols] = cat_imp.fit_transform(X_train[cat_cols])
    X_test_i[cat_cols] = cat_imp.transform(X_test[cat_cols])

    return X_train_i, X_test_i


def run_stability_analysis(X, y, k_values, do_impute, label):
    """25-run (5x5) repeated stratified CV. For each K, runs BOTH ANOVA and
    DODA, records selected feature sets, computes pairwise Jaccard similarity."""

    cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=42)
    all_jaccard = []

    for k in k_values:
        print(f"\n{'='*70}\n{label} STABILITY — TOP-{k}\n{'='*70}")

        for method in ["ANOVA", "DODA"]:
            selected_sets = []

            for train_idx, _ in cv.split(X, y):
                X_train = X.iloc[train_idx]
                y_train = y.iloc[train_idx]
                X_train_imp, _ = impute_if_needed(X_train, X_train, do_impute)

                if method == "ANOVA":
                    sel = SelectKBest(score_func=f_classif, k=k)
                    sel.fit(X_train_imp, y_train)
                    features = X_train_imp.columns[sel.get_support()].tolist()
                else:
                    operator = SklearnAdapter(SelectKBest(score_func=f_classif, k="all"))
                    provider = JSONProvider(WEIGHTS_FILE)
                    selector = DODASelector(operators=[operator], provider=provider,
                                             fusion=RankFusion(), top_k=k)
                    selector.fit(X_train_imp, y_train)
                    features = list(selector.get_selected_features())

                selected_sets.append(set(features))

            jaccard_scores = []
            for i in range(len(selected_sets)):
                for j in range(i + 1, len(selected_sets)):
                    inter = len(selected_sets[i] & selected_sets[j])
                    union = len(selected_sets[i] | selected_sets[j])
                    jaccard_scores.append(inter / union)

            for score in jaccard_scores:
                all_jaccard.append({"Top_K": k, "Method": method, "Jaccard": score})

            print(f"{method}: mean Jaccard = {np.mean(jaccard_scores):.4f} "
                  f"(std {np.std(jaccard_scores):.4f})")

    return pd.DataFrame(all_jaccard)


def run_cv_performance(X, y, k_values, do_impute, label):
    """25-run (5x5) repeated stratified CV predictive performance,
    ANOVA vs DODA, across 3 models and all Top-K values."""

    cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=42)
    results = []

    for run_id, (train_idx, test_idx) in enumerate(cv.split(X, y), start=1):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
        X_train_imp, X_test_imp = impute_if_needed(X_train, X_test, do_impute)

        for k in k_values:
            for method in ["ANOVA", "DODA"]:
                if method == "ANOVA":
                    sel = SelectKBest(score_func=f_classif, k=k)
                    X_train_sel = pd.DataFrame(sel.fit_transform(X_train_imp, y_train),
                                                columns=X_train_imp.columns[sel.get_support()])
                    X_test_sel = pd.DataFrame(sel.transform(X_test_imp),
                                               columns=X_train_sel.columns)
                else:
                    operator = SklearnAdapter(SelectKBest(score_func=f_classif, k="all"))
                    provider = JSONProvider(WEIGHTS_FILE)
                    selector = DODASelector(operators=[operator], provider=provider,
                                             fusion=RankFusion(), top_k=k)
                    X_train_sel = selector.fit_transform(X_train_imp, y_train)
                    X_test_sel = selector.transform(X_test_imp)

                models = make_models(y_train)
                for model_name, model in models.items():
                    if model_name == "LR":
                        scaler = StandardScaler()
                        Xtr = scaler.fit_transform(X_train_sel)
                        Xts = scaler.transform(X_test_sel)
                    else:
                        Xtr, Xts = X_train_sel, X_test_sel

                    model.fit(Xtr, y_train)
                    y_pred = model.predict(Xts)
                    y_prob = model.predict_proba(Xts)[:, 1]

                    results.append({
                        "Run": run_id, "Top_K": k, "Method": method, "Model": model_name,
                        "Accuracy": accuracy_score(y_test, y_pred),
                        "F1": f1_score(y_test, y_pred, zero_division=0),
                        "ROC_AUC": roc_auc_score(y_test, y_prob)
                    })

        if run_id % 5 == 0:
            print(f"{label}: completed {run_id}/25 CV runs")

    return pd.DataFrame(results)


def wilcoxon_holm_test(df, group_cols, value_col="Jaccard"):
    """Paired Wilcoxon signed-rank test (ANOVA vs DODA) + Cohen's d,
    with Holm correction across all comparisons in this dataframe."""
    rows = []
    for keys, group in df.groupby(group_cols):
        anova_vals = group[group.Method == "ANOVA"].sort_values(value_col)[value_col].values \
            if "Run" not in group.columns else \
            group[group.Method == "ANOVA"].sort_values("Run")[value_col].values
        doda_vals = group[group.Method == "DODA"].sort_values(value_col)[value_col].values \
            if "Run" not in group.columns else \
            group[group.Method == "DODA"].sort_values("Run")[value_col].values

        n = min(len(anova_vals), len(doda_vals))
        anova_vals, doda_vals = anova_vals[:n], doda_vals[:n]

        if np.allclose(anova_vals, doda_vals):
            stat, p = np.nan, 1.0
        else:
            try:
                stat, p = wilcoxon(anova_vals, doda_vals)
            except ValueError:
                stat, p = np.nan, 1.0

        diff = doda_vals - anova_vals
        pooled_std = np.std(np.concatenate([anova_vals, doda_vals]), ddof=1)
        cohens_d = diff.mean() / pooled_std if pooled_std > 0 else 0.0

        rows.append({
            **(dict(zip(group_cols, keys)) if isinstance(keys, tuple) else {group_cols[0]: keys}),
            "ANOVA_mean": anova_vals.mean(),
            "DODA_mean": doda_vals.mean(),
            "p_value": p,
            "cohens_d": cohens_d
        })

    result_df = pd.DataFrame(rows)
    if len(result_df) > 0:
        reject, p_adj, _, _ = multipletests(result_df["p_value"].fillna(1.0), method="holm")
        result_df["p_holm"] = p_adj
        result_df["significant"] = reject
    return result_df


print("Helper functions defined.")

# PRIMARY ANALYSIS (Imputed, n=400)

## 1. Baseline (80/20 split, for direct comparison with earlier notebooks)

In [ ]:
# =============================================================================
# PRIMARY: TRAIN-TEST SPLIT, IMPUTATION, SCALING
# =============================================================================

X_train, X_test, y_train, y_test = train_test_split(
    X_primary, y_primary, test_size=0.2, random_state=42, stratify=y_primary
)

X_train_imp, X_test_imp = impute_if_needed(X_train, X_test, do_impute=True)

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train_imp), columns=X_train_imp.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test_imp), columns=X_test_imp.columns)
y_train = y_train.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

print("PRIMARY split:", X_train_scaled.shape, X_test_scaled.shape)
print("Missing after imputation:", X_train_scaled.isnull().sum().sum(),
      X_test_scaled.isnull().sum().sum())

In [ ]:
# =============================================================================
# PRIMARY: ANOVA BASELINE TOP-K
# =============================================================================

k_values = [5, 10, 15, 20]

anova_selector = SelectKBest(score_func=f_classif, k="all")
anova_selector.fit(X_train_scaled, y_train)
anova_scores = pd.DataFrame({
    "Feature": X_train_scaled.columns, "ANOVA Score": anova_selector.scores_
}).sort_values("ANOVA Score", ascending=False).reset_index(drop=True)

anova_results = {}
for k in k_values:
    top_features = anova_scores.head(k)["Feature"].tolist()
    mask = X_train_scaled.columns.isin(top_features)
    anova_results[k] = {
        "features": top_features,
        "X_train": X_train_scaled.loc[:, mask],
        "X_test": X_test_scaled.loc[:, mask]
    }
    print(f"Top-{k}:", top_features)

In [ ]:
# =============================================================================
# PRIMARY: BASELINE MODEL EVALUATION
# =============================================================================

primary_baseline_results = []
for k in k_values:
    Xtr, Xts = anova_results[k]["X_train"], anova_results[k]["X_test"]
    models = make_models(y_train)
    for model_name, model in models.items():
        model.fit(Xtr, y_train)
        y_pred, y_prob = model.predict(Xts), model.predict_proba(Xts)[:, 1]
        primary_baseline_results.append({
            "Method": "ANOVA", "Top-K": k, "Model": model_name,
            "Accuracy": accuracy_score(y_test, y_pred),
            "F1 Score": f1_score(y_test, y_pred, zero_division=0),
            "ROC-AUC": roc_auc_score(y_test, y_prob)
        })

primary_baseline_df = pd.DataFrame(primary_baseline_results)
display(primary_baseline_df)

In [ ]:
# =============================================================================
# PRIMARY: RANK FUSION DODA
# =============================================================================

provider = JSONProvider(WEIGHTS_FILE)
fusion = RankFusion()

primary_doda_results_by_k = {}
for k in k_values:
    operator = SklearnAdapter(SelectKBest(score_func=f_classif, k="all"))
    selector = DODASelector(operators=[operator], provider=provider, fusion=fusion, top_k=k)
    X_train_sel = selector.fit_transform(X_train_scaled, y_train)
    X_test_sel = selector.transform(X_test_scaled)
    features = selector.get_selected_features()
    primary_doda_results_by_k[k] = {"X_train": X_train_sel, "X_test": X_test_sel,
                                     "features": features, "selector": selector}
    print(f"Top-{k}:", features)

In [ ]:
# =============================================================================
# PRIMARY: DODA MODEL EVALUATION + COMBINED BASELINE COMPARISON
# =============================================================================

primary_doda_results = []
for k in k_values:
    Xtr, Xts = primary_doda_results_by_k[k]["X_train"], primary_doda_results_by_k[k]["X_test"]
    models = make_models(y_train)
    for model_name, model in models.items():
        model.fit(Xtr, y_train)
        y_pred, y_prob = model.predict(Xts), model.predict_proba(Xts)[:, 1]
        primary_doda_results.append({
            "Method": "ANOVA + Rank Fusion", "Top-K": k, "Model": model_name,
            "Accuracy": accuracy_score(y_test, y_pred),
            "F1 Score": f1_score(y_test, y_pred, zero_division=0),
            "ROC-AUC": roc_auc_score(y_test, y_prob)
        })

primary_doda_df = pd.DataFrame(primary_doda_results)
primary_comparison_df = pd.concat([primary_baseline_df, primary_doda_df], ignore_index=True)
display(primary_comparison_df)

import os
os.makedirs("../../../results/ckd/primary", exist_ok=True)
primary_comparison_df.to_csv("../../../results/ckd/primary/anova_rankfusion_80_20_comparison.csv", index=False)

## 2. Feature-Selection Stability (25-run repeated CV)

In [ ]:
# =============================================================================
# PRIMARY: STABILITY ANALYSIS
# =============================================================================

primary_jaccard_df = run_stability_analysis(
    X_primary, y_primary, k_values, do_impute=True, label="PRIMARY"
)

primary_stability_summary = primary_jaccard_df.groupby(["Top_K", "Method"])["Jaccard"].agg(
    ["mean", "std"]
).reset_index()
print("\n" + "=" * 70)
print("PRIMARY STABILITY SUMMARY")
print("=" * 70)
display(primary_stability_summary)

In [ ]:
# =============================================================================
# PRIMARY: STABILITY SIGNIFICANCE TESTING (Wilcoxon + Holm + Cohen's d)
# =============================================================================

primary_stability_test = wilcoxon_holm_test(primary_jaccard_df, ["Top_K"], value_col="Jaccard")
print("=" * 70)
print("PRIMARY — ANOVA vs DODA STABILITY: SIGNIFICANCE TEST")
print("=" * 70)
display(primary_stability_test.round(4))

## 3. Predictive Performance (25-run repeated CV)

In [ ]:
# =============================================================================
# PRIMARY: CV PREDICTIVE PERFORMANCE
# =============================================================================

primary_cv_results = run_cv_performance(
    X_primary, y_primary, k_values, do_impute=True, label="PRIMARY"
)

primary_cv_summary = primary_cv_results.groupby(["Top_K", "Method", "Model"]).agg(
    {"Accuracy": ["mean", "std"], "F1": ["mean", "std"], "ROC_AUC": ["mean", "std"]}
).reset_index()
primary_cv_summary.columns = ["Top_K", "Method", "Model", "Acc_Mean", "Acc_STD",
                               "F1_Mean", "F1_STD", "AUC_Mean", "AUC_STD"]

os.makedirs("../../../results/ckd/primary", exist_ok=True)
primary_cv_results.to_csv("../../../results/ckd/primary/cv_performance_runs.csv", index=False)
primary_cv_summary.to_csv("../../../results/ckd/primary/cv_performance_summary.csv", index=False)

print("=" * 70)
print("PRIMARY CV PERFORMANCE SUMMARY")
print("=" * 70)
display(primary_cv_summary.round(4))

In [ ]:
# =============================================================================
# PRIMARY: PERFORMANCE SIGNIFICANCE TESTING
# =============================================================================

primary_perf_test = wilcoxon_holm_test(
    primary_cv_results, ["Top_K", "Model"], value_col="ROC_AUC"
)
print("=" * 70)
print("PRIMARY — ANOVA vs DODA ROC-AUC: SIGNIFICANCE TEST")
print("=" * 70)
display(primary_perf_test.round(4))

# SENSITIVITY ANALYSIS (Complete-Case, n=158)

Identical pipeline, applied to the 158 rows with zero missing values — no imputation step at all. If the Primary and Sensitivity conclusions agree, the imputation choice wasn't driving the results.

## 1. Baseline (80/20 split)

In [ ]:
# =============================================================================
# SENSITIVITY: TRAIN-TEST SPLIT + SCALING (no imputation needed)
# =============================================================================

Xs_train, Xs_test, ys_train, ys_test = train_test_split(
    X_sensitivity, y_sensitivity, test_size=0.2, random_state=42, stratify=y_sensitivity
)

scaler_s = StandardScaler()
Xs_train_scaled = pd.DataFrame(scaler_s.fit_transform(Xs_train), columns=Xs_train.columns)
Xs_test_scaled = pd.DataFrame(scaler_s.transform(Xs_test), columns=Xs_test.columns)
ys_train = ys_train.reset_index(drop=True)
ys_test = ys_test.reset_index(drop=True)

print("SENSITIVITY split:", Xs_train_scaled.shape, Xs_test_scaled.shape)

In [ ]:
# =============================================================================
# SENSITIVITY: ANOVA BASELINE TOP-K
# =============================================================================

anova_selector_s = SelectKBest(score_func=f_classif, k="all")
anova_selector_s.fit(Xs_train_scaled, ys_train)
anova_scores_s = pd.DataFrame({
    "Feature": Xs_train_scaled.columns, "ANOVA Score": anova_selector_s.scores_
}).sort_values("ANOVA Score", ascending=False).reset_index(drop=True)

anova_results_s = {}
for k in k_values:
    top_features = anova_scores_s.head(k)["Feature"].tolist()
    mask = Xs_train_scaled.columns.isin(top_features)
    anova_results_s[k] = {
        "features": top_features,
        "X_train": Xs_train_scaled.loc[:, mask],
        "X_test": Xs_test_scaled.loc[:, mask]
    }
    print(f"Top-{k}:", top_features)

In [ ]:
# =============================================================================
# SENSITIVITY: BASELINE MODEL EVALUATION
# =============================================================================

sensitivity_baseline_results = []
for k in k_values:
    Xtr, Xts = anova_results_s[k]["X_train"], anova_results_s[k]["X_test"]
    models = make_models(ys_train)
    for model_name, model in models.items():
        model.fit(Xtr, ys_train)
        y_pred, y_prob = model.predict(Xts), model.predict_proba(Xts)[:, 1]
        sensitivity_baseline_results.append({
            "Method": "ANOVA", "Top-K": k, "Model": model_name,
            "Accuracy": accuracy_score(ys_test, y_pred),
            "F1 Score": f1_score(ys_test, y_pred, zero_division=0),
            "ROC-AUC": roc_auc_score(ys_test, y_prob)
        })

sensitivity_baseline_df = pd.DataFrame(sensitivity_baseline_results)
display(sensitivity_baseline_df)

In [ ]:
# =============================================================================
# SENSITIVITY: RANK FUSION DODA + EVALUATION
# =============================================================================

sensitivity_doda_results_by_k = {}
for k in k_values:
    operator = SklearnAdapter(SelectKBest(score_func=f_classif, k="all"))
    selector = DODASelector(operators=[operator], provider=provider, fusion=RankFusion(), top_k=k)
    X_train_sel = selector.fit_transform(Xs_train_scaled, ys_train)
    X_test_sel = selector.transform(Xs_test_scaled)
    features = selector.get_selected_features()
    sensitivity_doda_results_by_k[k] = {"X_train": X_train_sel, "X_test": X_test_sel, "features": features}
    print(f"Top-{k}:", features)

sensitivity_doda_results = []
for k in k_values:
    Xtr, Xts = sensitivity_doda_results_by_k[k]["X_train"], sensitivity_doda_results_by_k[k]["X_test"]
    models = make_models(ys_train)
    for model_name, model in models.items():
        model.fit(Xtr, ys_train)
        y_pred, y_prob = model.predict(Xts), model.predict_proba(Xts)[:, 1]
        sensitivity_doda_results.append({
            "Method": "ANOVA + Rank Fusion", "Top-K": k, "Model": model_name,
            "Accuracy": accuracy_score(ys_test, y_pred),
            "F1 Score": f1_score(ys_test, y_pred, zero_division=0),
            "ROC-AUC": roc_auc_score(ys_test, y_prob)
        })

sensitivity_doda_df = pd.DataFrame(sensitivity_doda_results)
sensitivity_comparison_df = pd.concat([sensitivity_baseline_df, sensitivity_doda_df], ignore_index=True)
display(sensitivity_comparison_df)

os.makedirs("../../../results/ckd/sensitivity", exist_ok=True)
sensitivity_comparison_df.to_csv("../../../results/ckd/sensitivity/anova_rankfusion_80_20_comparison.csv", index=False)

## 2. Feature-Selection Stability (25-run repeated CV)

In [ ]:
# =============================================================================
# SENSITIVITY: STABILITY ANALYSIS
# =============================================================================

sensitivity_jaccard_df = run_stability_analysis(
    X_sensitivity, y_sensitivity, k_values, do_impute=False, label="SENSITIVITY"
)

sensitivity_stability_summary = sensitivity_jaccard_df.groupby(["Top_K", "Method"])["Jaccard"].agg(
    ["mean", "std"]
).reset_index()
print("\n" + "=" * 70)
print("SENSITIVITY STABILITY SUMMARY")
print("=" * 70)
display(sensitivity_stability_summary)

In [ ]:
sensitivity_stability_test = wilcoxon_holm_test(sensitivity_jaccard_df, ["Top_K"], value_col="Jaccard")
print("=" * 70)
print("SENSITIVITY — ANOVA vs DODA STABILITY: SIGNIFICANCE TEST")
print("=" * 70)
display(sensitivity_stability_test.round(4))

## 3. Predictive Performance (25-run repeated CV)

In [ ]:
# =============================================================================
# SENSITIVITY: CV PREDICTIVE PERFORMANCE
# =============================================================================

sensitivity_cv_results = run_cv_performance(
    X_sensitivity, y_sensitivity, k_values, do_impute=False, label="SENSITIVITY"
)

sensitivity_cv_summary = sensitivity_cv_results.groupby(["Top_K", "Method", "Model"]).agg(
    {"Accuracy": ["mean", "std"], "F1": ["mean", "std"], "ROC_AUC": ["mean", "std"]}
).reset_index()
sensitivity_cv_summary.columns = ["Top_K", "Method", "Model", "Acc_Mean", "Acc_STD",
                                   "F1_Mean", "F1_STD", "AUC_Mean", "AUC_STD"]

sensitivity_cv_results.to_csv("../../../results/ckd/sensitivity/cv_performance_runs.csv", index=False)
sensitivity_cv_summary.to_csv("../../../results/ckd/sensitivity/cv_performance_summary.csv", index=False)

print("=" * 70)
print("SENSITIVITY CV PERFORMANCE SUMMARY")
print("=" * 70)
display(sensitivity_cv_summary.round(4))

In [ ]:
sensitivity_perf_test = wilcoxon_holm_test(
    sensitivity_cv_results, ["Top_K", "Model"], value_col="ROC_AUC"
)
print("=" * 70)
print("SENSITIVITY — ANOVA vs DODA ROC-AUC: SIGNIFICANCE TEST")
print("=" * 70)
display(sensitivity_perf_test.round(4))

# Primary vs. Sensitivity — Head-to-Head Comparison

In [ ]:
# =============================================================================
# SIDE-BY-SIDE: STABILITY SUMMARY
# =============================================================================

primary_stability_summary["Analysis"] = "Primary (imputed, n=400)"
sensitivity_stability_summary["Analysis"] = "Sensitivity (complete-case, n=158)"

stability_side_by_side = pd.concat(
    [primary_stability_summary, sensitivity_stability_summary], ignore_index=True
).pivot_table(index=["Top_K", "Method"], columns="Analysis", values="mean").round(4)

print("=" * 70)
print("MEAN JACCARD STABILITY — PRIMARY vs SENSITIVITY")
print("=" * 70)
display(stability_side_by_side)

In [ ]:
# =============================================================================
# SIDE-BY-SIDE: PREDICTIVE PERFORMANCE (ROC-AUC) SUMMARY
# =============================================================================

primary_auc = primary_cv_summary[["Top_K", "Method", "Model", "AUC_Mean"]].copy()
primary_auc["Analysis"] = "Primary"
sensitivity_auc = sensitivity_cv_summary[["Top_K", "Method", "Model", "AUC_Mean"]].copy()
sensitivity_auc["Analysis"] = "Sensitivity"

auc_side_by_side = pd.concat([primary_auc, sensitivity_auc], ignore_index=True).pivot_table(
    index=["Top_K", "Method", "Model"], columns="Analysis", values="AUC_Mean"
).round(4)

print("=" * 70)
print("MEAN ROC-AUC — PRIMARY vs SENSITIVITY")
print("=" * 70)
display(auc_side_by_side)

auc_side_by_side.to_csv("../../../results/ckd/primary_vs_sensitivity_auc_comparison.csv")
stability_side_by_side.to_csv("../../../results/ckd/primary_vs_sensitivity_stability_comparison.csv")